# Reading a QR code from a photo

*graphics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/qr_code_decode)

Build a version 1–3 QR code from scratch (finder, timing and alignment patterns, BCH format bits, penalty-chosen mask, Reed–Solomon over GF(256)), photograph it with a random homography, blur, noise, uneven light and a thumb over it, and read it back as a tensor pipeline: adaptive threshold, matched filters for the 1:1:3:1:1 finders, DLT homography and grid_sample, BCH and Reed–Solomon decoding with $\mathrm{GF}(2^8)$ arithmetic as table gathers.

**Vector form**

$$\bar I = \tfrac{1}{k^2}\,\mathbf 1_{k\times k} \ast I,\ \ D = [\,I < \bar I - c\,] \qquad \mathbf x_{\text{img}} \simeq H\,\mathbf x_{\text{code}},\ \ \hat{\mathbf h} = \arg\min_{\lVert\mathbf h\rVert=1}\lVert A\mathbf h\rVert \qquad c(x) = d(x)\,x^{T} + \big(d(x)\,x^{T} \bmod g(x)\big),\ \ g(x) = \prod_{j=0}^{T-1}(x - \alpha^j)$$

**Index (tensor) form**

$$R_{bxy} = \min\!\Big(\max_s \sum_t K^{s}_t D^{\pm}_{b,x,y+t},\ \max_s \sum_t K^{s}_t D^{\pm}_{b,x+t,y}\Big) \qquad m_{bij} = D_b\big(H_{b}\,(j{+}\tfrac12,\ i{+}\tfrac12,\ 1)\big) \qquad S_{bj} = \bigoplus_i c_{bi} \otimes \alpha^{j(N-1-i)} \qquad e_{bj} = \bigoplus_i d_{bi} \otimes G_{ij} \qquad a \otimes b = \exp[\log a + \log b]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a \otimes b$ | `gf_mul(a, b) = EXP[LOG[a] + LOG[b]]` | broadcast | GF(256) product as two table gathers; $\alpha = 2$ generates all 255 non-zero bytes modulo $x^8+x^4+x^3+x^2+1$ (0x11D); $\oplus$ is XOR |
| $\bigoplus_i$ | `gf_sum(x, dim)` | (..., N) → (...) | XOR over the summed index $i$: add each of the 8 bit planes modulo 2 |
| $e_{bj} = \bigoplus_i d_{bi} \otimes G_{ij}$ | `rs_encode(data, n_ec)` | (B, k) → (B, T) | systematic Reed–Solomon: row $i$ of $G$ is $x^{T+k-1-i} \bmod g(x)$, so the ECC bytes are linear in the data |
| $S_{bj}$ | `syndromes(c, n_ec)` | (B, T) | block polynomial evaluated at $\alpha^j$; all zero exactly for codewords |
| $\Lambda(x),\ \Omega(x)$ | `berlekamp_massey(S), rs_decode(...)` | (B, T+1) | error locator by Berlekamp–Massey (batched with torch.where), roots by Chien search over all positions, magnitudes $e = X\,\Omega(X^{-1})/\Lambda'(X^{-1})$ (Forney) |
| $f_{15}$ | `format_code(ecl, mask), format_decode(bits)` | (B, 15) | BCH(15,5) with generator 0x537, XOR 0x5412; nearest of the 32 codewords fixes up to 3 bit errors |
| $D_{bxy}$ | `binarise(img)` | (B, H, W) | dark where the pixel is below its $31\times31$ box mean (conv2d) minus $c$: follows uneven lighting |
| $R_{bxy}$ | `finder_response(dark)` | (B, H, W) | matched filter $K^s$ for the 1:1:3:1:1 profile along rows and columns at 11 scales $s$; peaks are the finder centres (also checked along the diagonals) |
| $H_b$ | `dlt(code_points, img_points)` | (B, 3, 3) | DLT/SVD from 3 finder centres + the alignment pattern (version 1: parallelogram corner), then refined by gradient ascent on the fit to all known modules |
| $m_{bij}$ | `grid_sample(dark, H x)` | (B, n, n) | module $(i, j)$ read at $3\times3$ points inside it; then unmask (XOR, an involution) and read the zig-zag order |

Real readers such as ZXing follow the same outline: binarise with a local threshold, scan rows for the 1:1:3:1:1 run pattern of a finder and confirm it across columns and diagonals, pick three finders that form a right angle, estimate the module size and version from their spacing, look for an alignment pattern near where an affine guess puts it, fit a perspective transform, sample the grid, then decode the format and Reed–Solomon layers. Here the homography is also polished by maximising the correlation of the known modules (finders, timing, alignment, quiet zone) with the image through grid_sample, and the version is the one whose template fits best. The four error-correction levels L, M, Q, H restore roughly 7, 15, 25 and 30 % of the codewords: with $T$ ECC bytes a block corrects $\lfloor T/2 \rfloor$ unknown byte errors (twice as many if their positions were known, as erasures). A blob covering a share $f$ of the symbol hits more than $f$ of the codewords, because every byte is a $2\times4$ block of modules, so the success curves drop below those percentages. Masks only touch data modules and $M \oplus M = 0$, so unmasking is the same XOR; the encoder tries all 8 masks as one batch and keeps the lowest penalty (runs, $2\times2$ blocks, finder look-alikes, dark balance). Verification: the Reed–Solomon encoder reproduces the ECC bytes of the worked example in ISO/IEC 18004 ('01234567' as version 1-M) and the format words match the standard table; the rest is checked by round trips and zero syndromes. Simplifications: byte mode only, versions 1–3 (one alignment pattern, no version information blocks), no mirrored symbols, no erasure decoding. Version 1 has no alignment pattern, so strong perspective hurts it most. References: ISO/IEC 18004 (QR code symbology); I. S. Reed and G. Solomon, Polynomial codes over certain finite fields, J. SIAM 8 (1960) 300–304.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/qr_code_decode/main.py
"""Main loop: message → QR symbol → batch of synthetic photos → reader → success-rate sweep → animation."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation as eq
from . import plot, solve, tensors

IMAGE = 240  # photo size in pixels
MESSAGES = {
    "Reed-Solomon 1960": b"Reed-Solomon 1960",
    "Hello, tensors!": b"Hello, tensors!",
    "QR via PyTorch": b"QR via PyTorch",
    "x^8+x^4+x^3+x^2+1": b"x^8+x^4+x^3+x^2+1",
}
# ISO/IEC 18004 worked example: "01234567" as version 1-M (numeric mode), its 16 data and 10 ECC codewords.
ISO_DATA = [0x10, 0x20, 0x0C, 0x56, 0x61, 0x80] + [0xEC, 0x11] * 5
ISO_ECC = [0xA5, 0x24, 0xD4, 0xC1, 0xED, 0x36, 0xC7, 0x87, 0x2C, 0x55]


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    message, ecl = MESSAGES[params["message"]], params["ecc"]
    gen = torch.Generator().manual_seed(0)

    # 1. Encode, and check the pieces against known vectors.
    iso = eq.rs_encode(torch.tensor([ISO_DATA]), 10)[0].tolist()
    print(f"RS encoder on the ISO/IEC 18004 example '01234567' 1-M reproduces its 10 ECC bytes: {iso == ISO_ECC}")
    print(f"format word for level M, mask 0: {eq.format_code(0, 0):015b} (= 0x5412 mask, the standard value)")
    version = tensors.smallest_version(message, ecl, params["version"]) or 3
    if version != params["version"]:
        print(f"{len(message)} bytes do not fit version {params['version']}-{ecl}: using version {version}")
    sym = tensors.symbol(message, version, ecl)
    lay, n = tensors.layout(version, ecl), tensors.size(version)
    print(
        f"'{message.decode()}' ({len(message)} bytes) → version {version}-{ecl}: {n}×{n} modules, "
        f"{lay['blocks']} block(s) of {lay['k']} data + {lay['n_ec']} ECC bytes, corrects {lay['n_ec'] // 2} "
        f"byte errors per block"
    )
    print(f"mask penalties {sym['penalties'].tolist()} → mask {sym['mask']}")
    synd = eq.syndromes(sym["blocks"], lay["n_ec"])
    print(f"RS syndromes of the encoded blocks all zero: {bool((synd == 0).all())}")

    # 2. Photographs of it.
    count = 3 if fast else 5
    img, H_true = tensors.photos(sym["grid"], count, IMAGE, params["perspective"], params["occlusion"],
                                 params["noise"], gen)  # fmt: skip
    img = img.to(dev)

    # 3. Read them.
    tic = time.perf_counter()
    results = solve.decode(img, keep=True)
    print(f"decoded {count} photos ({IMAGE}×{IMAGE}, occlusion {params['occlusion']:.0%} of the symbol, "
          f"noise σ = {params['noise']}) in {time.perf_counter() - tic:.1f} s")  # fmt: skip
    for b, r in enumerate(results):
        if "modules" in r and r["version"] == version:
            r["module_errors"] = int((r["modules"] != sym["grid"]).sum())
        status = f"'{r['text']}'" if r["text"] is not None else f"failed ({r['stage']})"
        detail = ""
        if "byte_errors" in r:
            detail = (f"v{r['version']}-{r['ecl']} mask {r['mask']}, format bits fixed {r['format_errors']}, "
                      f"module errors {r.get('module_errors', '?')}, RS bytes fixed {r['byte_errors']}: ")  # fmt: skip
        print(f"  photo {b + 1}: {detail}{status}")
    ok = sum(r["text"] == message.decode() for r in results)
    print(f"{ok}/{count} photos read correctly")

    # 4. Success rate against occlusion, every ECC level, all photos of one level decoded as one batch.
    occs = [0.0, 0.16] if fast else [0.04 * i for i in range(8)]
    trials = 1 if fast else 5
    rates, versions = {}, {}
    tic = time.perf_counter()
    for level in tensors.LEVELS:
        v = tensors.smallest_version(message, level, params["version"]) or 3
        grid = tensors.symbol(message, v, level)["grid"]
        batch = torch.cat([tensors.photos(grid, trials, IMAGE, params["perspective"], o, params["noise"], gen)[0]
                           for o in occs])  # fmt: skip
        good = torch.tensor([r["text"] == message.decode() for r in solve.decode(batch.to(dev))])
        rates[level], versions[level] = good.view(len(occs), trials).float().mean(1), v
    print(f"success rate vs occlusion ({trials} photos per point, {time.perf_counter() - tic:.1f} s):")
    print("  occlusion " + " ".join(f"{o:5.0%}" for o in occs))
    for level in tensors.LEVELS:
        print(f"  {level} (v{versions[level]})    " + " ".join(f"{x:5.0%}" for x in rates[level].tolist()))

    block, place = tensors.codeword_map(version, ecl)
    return plot.animate(img.cpu(), results, sym, block, place, message.decode(), occs, rates, versions,
                        params["occlusion"], out_dir / "qr_code_decode.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/qr_code_decode/equation.py
r"""QR code → tensor translation: GF(256) arithmetic, Reed–Solomon and BCH codes, masks, homographies and filters.

GF(256). A byte a = (a₇ … a₀) is the polynomial Σ a_k x^k over GF(2), reduced modulo the primitive polynomial
p(x) = x⁸ + x⁴ + x³ + x² + 1 (0x11D). Addition is XOR. α = x = 2 has order 255, so every non-zero byte is α^l with
l = log a, and multiplication becomes two table look-ups (tensor gathers):

    a ⊗ b = exp[(log a + log b) mod 255],      a ⊘ b = exp[(log a − log b) mod 255].

A GF "matrix product" y_j = ⊕_i x_i ⊗ M_ij is an elementwise gf_mul followed by an XOR reduction over i (gf_sum): the
GF(256) analogue of einsum('i,ij->j') with the summed index i.

Reed–Solomon, QR flavour. A block of N bytes is the polynomial c(x) = Σ_i c_i x^{N−1−i} (first byte = highest power).
It is a codeword iff it is divisible by g(x) = Π_{j=0}^{T−1} (x − α^j), T = number of ECC bytes, i.e. iff all syndromes

    S_j = c(α^j) = ⊕_i c_i ⊗ V_ij,     V_ij = α^{j(N−1−i)}            (i summed with XOR)

vanish. Systematic encoding appends the remainder of d(x)·x^T mod g(x), which is linear in the k data bytes:

    e_j = ⊕_i d_i ⊗ G_ij,     row i of G = x^{T+k−1−i} mod g(x).

Format information: 5 bits (ECC level, mask) → BCH(15, 5) code with generator 0x537, XOR-ed with 0x5412. Any two of
its 32 codewords differ in at least 7 bits, so the nearest codeword corrects up to 3 bit errors.
"""

import functools

import torch
import torch.nn.functional as F

PRIMITIVE = 0x11D  # x^8 + x^4 + x^3 + x^2 + 1
FORMAT_GENERATOR = 0x537  # x^10 + x^8 + x^5 + x^4 + x^2 + x + 1
FORMAT_XOR = 0x5412


def gf_tables() -> tuple[torch.Tensor, torch.Tensor]:
    """exp (510,) with exp[l] = α^l (stored twice so exp[l1 + l2] needs no mod) and log (256,) with log[α^l] = l."""
    exp, log = torch.zeros(510, dtype=torch.long), torch.zeros(256, dtype=torch.long)
    x = 1
    for power in range(255):
        exp[power], log[x] = x, power
        x <<= 1  # multiply by α = x
        if x & 0x100:
            x ^= PRIMITIVE  # reduce modulo p(x)
    exp[255:] = exp[:255]
    return exp, log


EXP, LOG = gf_tables()


def gf_mul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """a ⊗ b elementwise (broadcasting): exp[log a + log b], and 0 if either factor is 0."""
    exp, log = EXP.to(a.device), LOG.to(a.device)
    return torch.where((a == 0) | (b == 0), 0, exp[log[a] + log[b]])


def gf_div(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """a ⊘ b elementwise for b ≠ 0: exp[(log a − log b) mod 255]."""
    exp, log = EXP.to(a.device), LOG.to(a.device)
    return torch.where(a == 0, 0, exp[(log[a] - log[b]) % 255])


def gf_pow_alpha(power: torch.Tensor) -> torch.Tensor:
    """α^power for integer tensors (negative powers allowed)."""
    return EXP.to(power.device)[power % 255]


def gf_sum(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """XOR-reduce along dim: split bytes into 8 bits, add each bit plane modulo 2, reassemble."""
    k = torch.arange(8, device=x.device)
    bits = (x.movedim(dim, -1)[..., None] >> k) & 1  # (..., N, 8)
    return ((bits.sum(-2) % 2) << k).sum(-1)


def gf_matmul(x: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    r"""y_{bj} = ⊕_i x_{bi} ⊗ M_{ij}   (i summed with XOR).   x (B, I), M (I, J) → (B, J)"""
    return gf_sum(gf_mul(x[:, :, None], M[None]), dim=1)


def gf_poly_eval(P: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""P_b(x_n) = ⊕_d P_{bd} ⊗ x_n^d for ascending coefficients P (B, D) at non-zero points x (N,) → (B, N)."""
    d = torch.arange(P.shape[1], device=P.device)
    powers = gf_pow_alpha(LOG.to(x.device)[x][:, None] * d)  # (N, D): x_n^d
    return gf_sum(gf_mul(P[:, None, :], powers[None]), dim=-1)


# --- Reed–Solomon over GF(256) ---------------------------------------------------------------------------------------


def rs_generator(n_ec: int) -> torch.Tensor:
    """g(x) = Π_{j<T} (x − α^j) as descending coefficients (T + 1,), g[0] = 1."""
    g = torch.ones(1, dtype=torch.long)
    for j in range(n_ec):
        zero = torch.zeros(1, dtype=torch.long)
        g = torch.cat([g, zero]) ^ torch.cat([zero, gf_mul(g, gf_pow_alpha(torch.tensor(j)))])  # g·x + g·α^j
    return g


def rs_generator_matrix(k: int, n_ec: int) -> torch.Tensor:
    """G (k, T): row i holds x^{T+k−1−i} mod g(x), descending, so ECC bytes = ⊕_i d_i ⊗ G_i (systematic encoding)."""
    g = rs_generator(n_ec)
    r = g[1:].clone()  # x^T ≡ g(x) − x^T = g[1:]   (minus = plus in GF(2^8))
    rows = [r]
    for _ in range(k - 1):  # x·r mod g: shift left and fold the overflowing coefficient back in
        lead, r = r[0], torch.cat([r[1:], torch.zeros(1, dtype=torch.long)])
        r = r ^ gf_mul(lead, g[1:])
        rows.append(r)
    return torch.stack(rows[::-1])  # row i ↔ power T + k − 1 − i


def rs_encode(data: torch.Tensor, n_ec: int) -> torch.Tensor:
    r"""e_{bj} = ⊕_i d_{bi} ⊗ G_{ij}.   data (B, k) bytes → ECC bytes (B, T)."""
    return gf_matmul(data, rs_generator_matrix(data.shape[1], n_ec).to(data.device))


def syndrome_matrix(n: int, n_ec: int, device=None) -> torch.Tensor:
    """V (N, T) with V_ij = α^{j (N−1−i)}: column j evaluates the block polynomial at α^j."""
    i, j = torch.arange(n, device=device)[:, None], torch.arange(n_ec, device=device)[None]
    return gf_pow_alpha(j * (n - 1 - i))


def syndromes(c: torch.Tensor, n_ec: int) -> torch.Tensor:
    r"""S_{bj} = ⊕_i c_{bi} ⊗ α^{j(N−1−i)}.   c (B, N) → (B, T); all zero ⇔ c is a codeword."""
    return gf_matmul(c, syndrome_matrix(c.shape[1], n_ec, c.device))


# --- format information: BCH(15, 5) ---------------------------------------------------------------------------------


def format_code(ecl_bits: int, mask: int) -> int:
    """15-bit format word: data = (ECC level bits, mask) · x^10 + remainder mod 0x537, XOR 0x5412."""
    data = (ecl_bits << 3) | mask
    rem = data << 10
    for bit in range(14, 9, -1):  # long division by the degree-10 generator
        if rem >> bit & 1:
            rem ^= FORMAT_GENERATOR << (bit - 10)
    return ((data << 10) | rem) ^ FORMAT_XOR


def format_table() -> torch.Tensor:
    """All 32 format codewords as bits (32, 15), bit i = coefficient of x^i; row index = (ECC level bits) · 8 + mask."""
    words = torch.tensor([format_code(d >> 3, d & 7) for d in range(32)])
    return (words[:, None] >> torch.arange(15)) & 1


def format_decode(bits: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Nearest valid codeword by Hamming distance. bits (B, 15) → index (B,) = ecl·8 + mask, bit errors (B,)."""
    dist = (bits[:, None, :] != format_table().to(bits.device)[None]).sum(-1)  # (B, 32)
    errors, index = dist.min(dim=1)
    return index, errors


# --- masks and the penalty score -------------------------------------------------------------------------------------


def mask_pattern(k: int, n: int, device=None) -> torch.Tensor:
    """Mask k of the standard as an (n, n) bool grid; i = row, j = column. Flipping data modules where it is True."""
    i, j = torch.meshgrid(torch.arange(n, device=device), torch.arange(n, device=device), indexing="ij")
    rule = [
        (i + j) % 2 == 0,
        i % 2 == 0,
        j % 3 == 0,
        (i + j) % 3 == 0,
        (i // 2 + j // 3) % 2 == 0,
        (i * j) % 2 + (i * j) % 3 == 0,
        ((i * j) % 2 + (i * j) % 3) % 2 == 0,
        ((i + j) % 2 + (i * j) % 3) % 2 == 0,
    ]
    return rule[k]


def penalty(grid: torch.Tensor) -> torch.Tensor:
    """Standard mask penalty N1 + N2 + N3 + N4 for a batch of symbols grid (K, n, n) of 0/1 → (K,).

    N1: each run of ≥ 5 equal modules in a row/column scores 3 + (length − 5) = (windows of 5 in it) + 2 per run.
    N2: 3 per 2 × 2 block of one colour. N3: 40 per finder-like 1:1:3:1:1 pattern with 4 light modules on one side
    (counted by window matching, outside the symbol counts as light). N4: 10 per 5 % the dark share is off 50 %."""
    total = torch.zeros(grid.shape[0], dtype=torch.long, device=grid.device)
    pattern = torch.tensor([1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0], device=grid.device)
    for g in (grid, grid.transpose(1, 2)):  # rows, then columns
        win = g.unfold(2, 5, 1)  # (K, n, n−4, 5)
        same = (win == win[..., :1]).all(-1)
        starts = torch.ones_like(same)
        starts[..., 1:] = g[..., :-5] != g[..., 1:-4]  # run begins at window start
        total += same.sum((1, 2)) + 2 * (same & starts).sum((1, 2))
        padded = F.pad(g, (4, 4))
        win11 = padded.unfold(2, 11, 1)  # (K, n, n−2, 11)
        hits = (win11 == pattern).all(-1) | (win11 == pattern.flip(0)).all(-1)
        total += 40 * hits.sum((1, 2))
    blocks = (grid[:, :-1, :-1] == grid[:, 1:, :-1]) & (grid[:, :-1, :-1] == grid[:, :-1, 1:])
    total += 3 * (blocks & (grid[:, :-1, :-1] == grid[:, 1:, 1:])).sum((1, 2))
    dark = grid.float().mean((1, 2))
    total += 10 * ((100 * dark - 50).abs() // 5).long()
    return total


# --- projective geometry ---------------------------------------------------------------------------------------------


def apply_homography(H: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'_{bni} = H_{bij} x_{bnj} / (H_{b3j} x_{bnj})   (j summed).   H (B, 3, 3), x (B, N, 2) → (B, N, 2)"""
    xh = torch.cat([x, torch.ones_like(x[..., :1])], -1)
    y = torch.einsum("bij,bnj->bni", H, xh)
    return y[..., :2] / y[..., 2:]


def dlt(src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:
    r"""Direct linear transform: h = argmin_{|h|=1} |A h| by SVD, two rows of A per match from x' × (H x) = 0.

    src, dst (B, N ≥ 4, 2) → H (B, 3, 3) with H₃₃ = 1. Points are first scaled to unit size (Hartley normalisation)."""

    def normaliser(x):
        centre, scale = x.mean(1), 2**0.5 / (x - x.mean(1, keepdim=True)).norm(dim=-1).mean(1)
        T = torch.zeros(x.shape[0], 3, 3, dtype=x.dtype, device=x.device)
        T[:, 0, 0] = T[:, 1, 1] = scale
        T[:, :2, 2] = -scale[:, None] * centre
        T[:, 2, 2] = 1
        return T

    T1, T2 = normaliser(src), normaliser(dst)
    a, b = apply_homography(T1, src), apply_homography(T2, dst)
    u, v, u2, v2 = a[..., 0], a[..., 1], b[..., 0], b[..., 1]
    one, zero = torch.ones_like(u), torch.zeros_like(u)
    rows_u = torch.stack([-u, -v, -one, zero, zero, zero, u2 * u, u2 * v, u2], -1)  # (B, N, 9)
    rows_v = torch.stack([zero, zero, zero, -u, -v, -one, v2 * u, v2 * v, v2], -1)
    A = torch.cat([rows_u, rows_v], 1)  # (B, 2N, 9)
    H = torch.linalg.svd(A)[2][:, -1].reshape(-1, 3, 3)  # right singular vector of the smallest singular value
    H = torch.linalg.solve(T2, H @ T1)
    return H / H[:, 2:, 2:]


# --- image filters ---------------------------------------------------------------------------------------------------


def box_mean(img: torch.Tensor, k: int) -> torch.Tensor:
    r"""Local mean \bar I_{bxy} = (1/k²) Σ_{|u|,|v| ≤ k/2} I_{b,x+u,y+v}: a k × k box filter, done as two 1D
    conv2d passes (a row box, then a column box).  (B, H, W) → same"""
    kernel = torch.full((1, 1, 1, k), 1.0 / k, device=img.device)
    x = F.conv2d(F.pad(img[:, None], (k // 2, k // 2, 0, 0), mode="replicate"), kernel)
    return F.conv2d(F.pad(x, (0, 0, k // 2, k // 2), mode="replicate"), kernel.transpose(2, 3))[:, 0]


@functools.cache
def profile_kernel(scale: float, edges: tuple, signs: tuple, slack: float = 0.25) -> torch.Tensor:
    """1D matched filter for a run-length profile: tap t gets the sign of the band |t|/scale falls in, or 0 within
    `slack` modules of a band edge (so blur and a slightly wrong scale cost little); normalised so a perfect match
    gives 1. Finder 1:1:3:1:1 (dark, light, dark, light, dark): edges (1.5, 2.5, 3.5), signs (+1, −1, +1) (|t| in
    modules from the centre). Cached per scale."""
    half = int(edges[-1] * scale)
    u = torch.arange(-half, half + 1, dtype=torch.float32).abs() / scale
    band = torch.bucketize(u, torch.tensor(edges[:-1]), right=True).clamp(max=len(signs) - 1)
    near_edge = (u[:, None] - torch.tensor(edges)).abs().min(1).values < slack
    k = torch.where(near_edge, 0.0, torch.tensor(signs, dtype=torch.float32)[band])
    return k / k.abs().sum()


def line_responses(d: torch.Tensor, kernel: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Correlate the ±1 image d (B, H, W) with a 1D profile along rows and along columns:
    r^h_{bxy} = Σ_t K_t d_{b,x,y+t},  r^v_{bxy} = Σ_t K_t d_{b,x+t,y}.  Outside the image counts as light (−1)."""
    half = (len(kernel) - 1) // 2
    padded = F.pad(d[:, None], (half,) * 4, value=-1.0)
    kh = kernel.to(d.device).view(1, 1, 1, -1)
    rh = F.conv2d(padded[..., half:-half, :], kh)[:, 0]
    rv = F.conv2d(padded[..., half:-half], kh.transpose(2, 3))[:, 0]
    return rh, rv


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/qr_code_decode/tensors.py
"""Build the concrete tensors: the QR symbol (function patterns, codewords, placement, mask) and synthetic photos of it.

Grid convention: module (i, j) is row i, column j, with 1 = dark. In the code plane the module occupies
X ∈ [j, j + 1), Y ∈ [i, i + 1) (module units); the quiet zone is the 4-module light border around it.
Pixel convention: pixel (column u, row v) has its centre at image coordinates (u, v).
"""

import math

import torch
import torch.nn.functional as F

from . import equation as eq

# ISO/IEC 18004 tables for versions 1–3 (one or two equal blocks, so interleaving is a transpose).
TOTAL_CODEWORDS = {1: 26, 2: 44, 3: 70}
ECC_PER_BLOCK = {"L": (7, 10, 15), "M": (10, 16, 26), "Q": (13, 22, 18), "H": (17, 28, 22)}
BLOCKS = {"L": (1, 1, 1), "M": (1, 1, 1), "Q": (1, 1, 2), "H": (1, 1, 2)}
ECL_BITS = {"L": 1, "M": 0, "Q": 3, "H": 2}  # the 2 bits stored in the format information
LEVELS = ["L", "M", "Q", "H"]
QUIET = 4


def size(version: int) -> int:
    return 17 + 4 * version


def layout(version: int, ecl: str) -> dict:
    """Block structure: number of blocks, data bytes k and ECC bytes T per block, byte-mode capacity."""
    nb, n_ec = BLOCKS[ecl][version - 1], ECC_PER_BLOCK[ecl][version - 1]
    k = TOTAL_CODEWORDS[version] // nb - n_ec
    return {"blocks": nb, "k": k, "n_ec": n_ec, "capacity": (8 * nb * k - 12) // 8}  # 4 mode + 8 count bits


def smallest_version(message: bytes, ecl: str, at_least: int = 1) -> int | None:
    fits = [v for v in range(max(1, at_least), 4) if len(message) <= layout(v, ecl)["capacity"]]
    return fits[0] if fits else None


def format_positions(n: int) -> tuple[torch.Tensor, torch.Tensor]:
    """(row, col) of format bit i (bit 0 = x^0) for the copy around the top-left finder and the split second copy."""
    first = [(i, 8) for i in range(6)] + [(7, 8), (8, 8), (8, 7)] + [(8, 14 - i) for i in range(9, 15)]
    second = [(8, n - 1 - i) for i in range(8)] + [(n - 15 + i, 8) for i in range(8, 15)]
    return torch.tensor(first), torch.tensor(second)


def function_patterns(version: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Fixed modules: values (n, n) and a mask (n, n) of modules that are not data (finders + separators, timing,
    alignment, format areas, the dark module). Format bits are left light here and drawn by draw_format."""
    n = size(version)
    val = torch.zeros(n, n, dtype=torch.long)
    fun = torch.zeros(n, n, dtype=torch.bool)
    val[6, :] = val[:, 6] = (torch.arange(n) % 2 == 0).long()  # timing patterns
    fun[6, :] = fun[:, 6] = True
    d = torch.arange(-4, 5)
    ring = torch.maximum(d.abs()[:, None], d.abs()[None])  # Chebyshev distance from the centre
    for r, c in [(3, 3), (3, n - 4), (n - 4, 3)]:  # finder 7 × 7 plus its light separator
        rows, cols = r + d, c + d
        ok_r, ok_c = (rows >= 0) & (rows < n), (cols >= 0) & (cols < n)
        sub = ((ring != 2) & (ring != 4)).long()[ok_r][:, ok_c]
        val[rows[ok_r][:, None], cols[ok_c][None]] = sub
        fun[rows[ok_r][:, None], cols[ok_c][None]] = True
    if version >= 2:  # one alignment pattern centred at (n − 7, n − 7)
        a = n - 7
        val[a - 2 : a + 3, a - 2 : a + 3] = (ring[2:7, 2:7] != 1).long()
        fun[a - 2 : a + 3, a - 2 : a + 3] = True
    for pos in format_positions(n):
        fun[pos[:, 0], pos[:, 1]] = True
    val[n - 8, 8], fun[n - 8, 8] = 1, True  # the dark module
    return val, fun


def data_positions(fun: torch.Tensor) -> torch.Tensor:
    """(row, col) of every data module in placement order (M, 2): two-module-wide columns from the right, zig-zagging
    up and down, skipping the vertical timing column 6."""
    n = fun.shape[0]
    order = []
    right = n - 1
    while right >= 1:
        if right == 6:
            right = 5
        upward = ((right + 1) & 2) == 0
        for step in range(n):
            row = n - 1 - step if upward else step
            for col in (right, right - 1):
                if not fun[row, col]:
                    order.append((row, col))
        right -= 2
    return torch.tensor(order)


def draw_format(grid: torch.Tensor, ecl: str, mask: int) -> torch.Tensor:
    """Write both copies of the 15 format bits into grid (..., n, n)."""
    word = eq.format_code(ECL_BITS[ecl], mask)
    bits = (torch.tensor(word) >> torch.arange(15)) & 1
    grid = grid.clone()
    for pos in format_positions(grid.shape[-1]):
        grid[..., pos[:, 0], pos[:, 1]] = bits
    return grid


def data_codewords(message: bytes, version: int, ecl: str) -> torch.Tensor:
    """Byte-mode bit stream: mode 0100, 8-bit length, the bytes, terminator, then pad bytes 0xEC 0x11 → (nb·k,)."""
    lay = layout(version, ecl)
    n_data = lay["blocks"] * lay["k"]
    bits = [0, 1, 0, 0] + [(len(message) >> s) & 1 for s in range(7, -1, -1)]
    bits += [(byte >> s) & 1 for byte in message for s in range(7, -1, -1)]
    bits += [0] * min(4, 8 * n_data - len(bits))  # terminator
    bits += [0] * (-len(bits) % 8)
    words = [int("".join(map(str, bits[i : i + 8])), 2) for i in range(0, len(bits), 8)]
    words += [0xEC, 0x11] * ((n_data - len(words)) // 2 + 1)
    return torch.tensor(words[:n_data])


def codewords(message: bytes, version: int, ecl: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Data split into blocks D (nb, k), ECC bytes E (nb, T) by Reed–Solomon, interleaved column by column."""
    lay = layout(version, ecl)
    D = data_codewords(message, version, ecl).reshape(lay["blocks"], lay["k"])
    E = eq.rs_encode(D, lay["n_ec"])
    return torch.cat([D.T.reshape(-1), E.T.reshape(-1)]), torch.cat([D, E], 1)


def symbol(message: bytes, version: int, ecl: str, mask: int | None = None) -> dict:
    """The finished QR symbol. mask=None → try all 8 masks at once and keep the lowest penalty."""
    val, fun = function_patterns(version)
    pos = data_positions(fun)
    stream, blocks = codewords(message, version, ecl)
    bits = ((stream[:, None] >> torch.arange(7, -1, -1)) & 1).reshape(-1)  # MSB first
    raw = val.clone()
    raw[pos[: len(bits), 0], pos[: len(bits), 1]] = bits  # remainder bits stay 0
    n = size(version)
    masks = torch.stack([eq.mask_pattern(k, n) for k in range(8)]) & ~fun  # data modules only
    candidates = torch.stack([draw_format(raw ^ masks[k].long(), ecl, k) for k in range(8)])  # (8, n, n)
    scores = eq.penalty(candidates)
    chosen = int(scores.argmin()) if mask is None else mask
    return {
        "grid": candidates[chosen],
        "unmasked": raw,
        "function": fun,
        "positions": pos,
        "mask": chosen,
        "penalties": scores,
        "blocks": blocks,
        "version": version,
        "ecl": ecl,
    }


# --- synthetic photos ------------------------------------------------------------------------------------------------


def homographies(n: int, count: int, image: int, perspective: float, gen: torch.Generator) -> torch.Tensor:
    """Random code-plane → image homographies (B, 3, 3): rotate by any angle, scale the symbol to about 60 % of the
    frame, then push each corner by up to perspective × side in a random direction and solve the DLT."""
    corners = torch.tensor([[0.0, 0.0], [n, 0.0], [n, n], [0.0, n]])
    theta = 2 * math.pi * torch.rand(count, generator=gen)
    side = image * 0.6 * (0.9 + 0.15 * torch.rand(count, generator=gen))
    rot = torch.stack([torch.cos(theta), -torch.sin(theta), torch.sin(theta), torch.cos(theta)], -1).view(-1, 2, 2)
    centred = (corners - n / 2) / n  # (4, 2)
    shift = image / 2 + 0.04 * image * (2 * torch.rand(count, 1, 2, generator=gen) - 1)
    img_corners = shift + side[:, None, None] * torch.einsum("bij,cj->bci", rot, centred)
    img_corners = img_corners + perspective * side[:, None, None] * (2 * torch.rand(count, 4, 2, generator=gen) - 1)
    return eq.dlt(corners.expand(count, 4, 2), img_corners)


def render(grid: torch.Tensor, H: torch.Tensor, image: int, occlusion: float, noise: float, gen: torch.Generator):
    """Photos (B, image, image) in [0, 1] of the symbol under homographies H (B, 3, 3), with a textured table, an
    occluding blob of area occlusion × n² placed over the symbol, uneven lighting, blur and pixel noise."""
    n, B = grid.shape[0], H.shape[0]
    ss = 2  # supersampling
    s = (torch.arange(ss * image) + 0.5) / ss - 0.5
    vv, uu = torch.meshgrid(s, s, indexing="ij")
    pix = torch.stack([uu, vv], -1).reshape(1, -1, 2).expand(B, -1, -1)
    XY = eq.apply_homography(torch.linalg.inv(H), pix)  # (B, P, 2) code-plane point seen by every sub-pixel
    X, Y = XY[..., 0], XY[..., 1]
    col, row = X.floor().long(), Y.floor().long()
    inside = (col >= 0) & (col < n) & (row >= 0) & (row < n)
    module = grid[row.clamp(0, n - 1), col.clamp(0, n - 1)] * inside
    card = (X >= -QUIET) & (X < n + QUIET) & (Y >= -QUIET) & (Y < n + QUIET)
    table = (
        0.42
        + 0.06 * torch.sin(0.09 * pix[..., 0] + 0.5 * torch.sin(0.05 * pix[..., 1]))
        + 0.04 * torch.sin(0.4 * pix[..., 1])
    )
    img = torch.where(card, 0.92 - 0.8 * module, table)
    # Occluding blob: an ellipse over the data region (a thumb or a sticker), dark or light.
    r = torch.exp(0.3 * (2 * torch.rand(B, generator=gen) - 1))  # aspect ratio
    a = n * (occlusion / math.pi) ** 0.5
    centre = n * (0.35 + 0.3 * torch.rand(B, 2, generator=gen))
    phi = math.pi * torch.rand(B, generator=gen)
    dx, dy = X - centre[:, :1], Y - centre[:, 1:]
    p = dx * torch.cos(phi)[:, None] + dy * torch.sin(phi)[:, None]
    q = -dx * torch.sin(phi)[:, None] + dy * torch.cos(phi)[:, None]
    blob = (p / (a * r[:, None])) ** 2 + (q * r[:, None] / a) ** 2 < 1
    shade = torch.where(torch.rand(B, generator=gen) < 0.5, 0.18, 0.8)[:, None]
    if occlusion > 0:
        img = torch.where(blob, shade, img)
    img = img.reshape(B, image, ss, image, ss).mean((2, 4))
    # Uneven lighting: a random linear gradient times a vignette.
    t = torch.linspace(-1, 1, image)
    gy, gx = torch.meshgrid(t, t, indexing="ij")
    ang = 2 * math.pi * torch.rand(B, 1, 1, generator=gen)
    light = (0.8 + 0.3 * (gx * torch.cos(ang) + gy * torch.sin(ang))) * (1 - 0.18 * (gx**2 + gy**2))
    img = img * light
    taps = torch.exp(-0.5 * (torch.arange(-2, 3.0) / 0.9) ** 2)  # Gaussian blur σ = 0.9 px, separable
    taps = taps / taps.sum()
    img = F.conv2d(F.pad(img[:, None], (2, 2, 0, 0), mode="replicate"), taps.view(1, 1, 1, 5))
    img = F.conv2d(F.pad(img, (0, 0, 2, 2), mode="replicate"), taps.view(1, 1, 5, 1))[:, 0]
    img = img + noise * torch.randn(img.shape, generator=gen)
    return img.clamp(0, 1)


def photos(grid, count, image, perspective, occlusion, noise, gen, chunk: int = 8):
    """Render count photos in chunks (memory). Returns images (B, image, image) and the true homographies."""
    H = homographies(grid.shape[0], count, image, perspective, gen)
    imgs = [render(grid, H[i : i + chunk], image, occlusion, noise, gen) for i in range(0, count, chunk)]
    return torch.cat(imgs), H


def codeword_map(version: int, ecl: str) -> tuple[torch.Tensor, torch.Tensor]:
    """For every module, the (block, byte position) its bit belongs to after de-interleaving; −1 for modules that
    carry no codeword (function patterns, remainder bits). → two (n, n) long tensors."""
    lay = layout(version, ecl)
    nb, k = lay["blocks"], lay["k"]
    _, fun = function_patterns(version)
    pos = data_positions(fun)[: 8 * TOTAL_CODEWORDS[version]]
    byte = torch.arange(len(pos)) // 8  # index in the interleaved stream
    is_ecc = byte >= nb * k
    rel = torch.where(is_ecc, byte - nb * k, byte)
    n = size(version)
    block, place = torch.full((n, n), -1), torch.full((n, n), -1)
    block[pos[:, 0], pos[:, 1]] = rel % nb
    place[pos[:, 0], pos[:, 1]] = torch.where(is_ecc, k + rel // nb, rel // nb)
    return block, place


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/qr_code_decode/solve.py
"""The reader: binarise → find the finders → homography → sample modules → format → unmask → Reed–Solomon → text.

Image-wide steps (thresholding, matched filtering, sampling, Reed–Solomon) run on the whole batch of photos at once;
picking peaks and fitting each photo's homography is a short loop over photos.
"""

import itertools

import torch
import torch.nn.functional as F

from . import equation as eq
from . import tensors

FINDER = ((1.5, 2.5, 3.5), (1, -1, 1))  # 1:1:3:1:1 dark-light-dark-light-dark, band edges in modules from centre
SCALES = [2.5 * 1.15**k for k in range(11)]  # 2.5 … 10 pixels per module tried by the finder search


# --- 1. binarise -----------------------------------------------------------------------------------------------------


def binarise(img: torch.Tensor, window: int = 31, offset: float = 0.03) -> torch.Tensor:
    """Adaptive threshold: dark where the (lightly smoothed) pixel is below its local box mean minus an offset.
    img (B, H, W) → (B, H, W) bool. The local mean follows uneven lighting, a global threshold would not."""
    smooth = eq.box_mean(img, 3)
    return smooth < eq.box_mean(img, window) - offset


# --- 2. finder patterns ----------------------------------------------------------------------------------------------


def finder_response(dark: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """R_{bxy} = min(max_s r^h_s, max_s r^v_s): the 1:1:3:1:1 profile must fit along the row and along the column
    through (x, y), each at its best scale s (pixels per module; perspective makes them differ). Along any line
    through its centre a finder keeps these ratios, whatever the rotation. dark (B, H, W) → R in [−1, 1]
    (1 = perfect match) and the mean of the two best scales, both (B, H, W)."""
    d = dark.float() * 2 - 1  # dark +1, light −1
    best_h, best_v = torch.full_like(d, -1.0), torch.full_like(d, -1.0)
    scale_h, scale_v = torch.zeros_like(d), torch.zeros_like(d)
    for s in SCALES:
        rh, rv = eq.line_responses(d, eq.profile_kernel(s, *FINDER))
        scale_h, scale_v = torch.where(rh > best_h, s, scale_h), torch.where(rv > best_v, s, scale_v)
        best_h, best_v = torch.maximum(best_h, rh), torch.maximum(best_v, rv)
    return torch.minimum(best_h, best_v), (scale_h + scale_v) / 2


def peaks(R, pooled, scale, dark, radius: int = 7, count: int = 8, floor: float = 0.55) -> torch.Tensor:
    """Local maxima of one response map R (H, W) (where R equals its max-pooled version `pooled`) above floor →
    (k, 4) rows (x, y, response, scale), where the response also includes the two diagonals (diagonal_response).
    Each position is refined to the centroid of the plateau ≥ 90 % of its peak (a finder centre is a 3 × 3-module
    plateau, so the argmax alone would sit on its edge)."""
    ys, xs = torch.nonzero((R == pooled) & (R > floor), as_tuple=True)
    vals = R[ys, xs]
    order = vals.argsort(descending=True)[: 4 * count]
    out = []
    h, w = R.shape
    for i in order.tolist():
        x, y, v = xs[i].item(), ys[i].item(), vals[i].item()
        if any((x - o[0]) ** 2 + (y - o[1]) ** 2 < radius**2 for o in out):
            continue  # plateau ties: keep one maximum per blob
        y0, y1, x0, x1 = max(0, y - radius), min(h, y + radius + 1), max(0, x - radius), min(w, x + radius + 1)
        patch = R[y0:y1, x0:x1]
        weight = (patch >= 0.9 * v).float()
        gy, gx = torch.meshgrid(torch.arange(y0, y1), torch.arange(x0, x1), indexing="ij")
        cx, cy = float((gx * weight).sum() / weight.sum()), float((gy * weight).sum() / weight.sum())
        out.append([cx, cy, min(v, diagonal_response(dark, cx, cy)), float(scale[y, x])])
        if len(out) == count:
            break
    return torch.tensor(out).reshape(-1, 4)


def diagonal_response(dark: torch.Tensor, x: float, y: float) -> float:
    """The 1:1:3:1:1 match along both diagonals through (x, y), best scale each, the smaller of the two. Finders
    pass in every direction; the odd cross of data modules that fools the row/column test seldom does."""
    h, w = dark.shape
    kernels = [eq.profile_kernel(s, *FINDER) for s in SCALES]
    half = (len(kernels[-1]) - 1) // 2
    K = torch.stack([F.pad(k, (half - len(k) // 2,) * 2) for k in kernels])  # (S, 2·half + 1), zero-padded
    t = torch.arange(-half, half + 1) / 2**0.5  # one-pixel steps along a diagonal
    best = []
    for dy in (1, -1):
        u, v = (x + t).round().long(), (y + dy * t).round().long()
        inside = (u >= 0) & (u < w) & (v >= 0) & (v < h)
        d = torch.where(inside, dark[v.clamp(0, h - 1), u.clamp(0, w - 1)].float() * 2 - 1, -1.0)
        best.append(float((K @ d).max()))
    return min(best)


def pick_finders(cands: torch.Tensor) -> tuple[torch.Tensor | None, float]:
    """Choose the three strongest candidates that form a right isosceles triangle, ordered top-left (the right-angle
    corner), top-right, bottom-left by the sign of the cross product. The finders of a version 1–3 symbol are
    14–22 modules apart and match at similar scales. cands (k, 4) rows (x, y, response, scale) → (3, 2) or None,
    and the mean scale (pixels per module)."""
    best, best_score, module = None, -1e9, 0.0
    for tri in itertools.combinations(range(len(cands)), 3):
        p, resp, scales = cands[list(tri), :2], cands[list(tri), 2], cands[list(tri), 3]
        if scales.max() > 1.8 * scales.min():
            continue
        for corner in range(3):  # perspective can make any vertex look like the right angle: try all three
            tl, p1, p2 = p[corner], *p[[k for k in range(3) if k != corner]]
            e1, e2 = p1 - tl, p2 - tl
            if e1[0] * e2[1] - e1[1] * e2[0] < 0:  # image y points down: TL→TR × TL→BL > 0
                p1, p2, e1, e2 = p2, p1, e2, e1
            l1, l2 = e1.norm(), e2.norm()
            if min(l1, l2) < 10 * scales.mean() or max(l1, l2) > 28 * scales.mean():
                continue
            error = float((e1 @ e2).abs() / (l1 * l2) + (l1 - l2).abs() / torch.maximum(l1, l2))  # 0: right, equal
            if error > 0.6:
                continue
            score = float(resp.mean()) - 0.25 * error
            if score > best_score:
                best_score, best, module = score, torch.stack([tl, p1, p2]), float(scales.mean())
    return best, module


# --- 3. homography ---------------------------------------------------------------------------------------------------


def find_alignment(dark: torch.Tensor, guess: torch.Tensor, ex: torch.Tensor, ey: torch.Tensor):
    """2D matched filter for the 5 × 5-module alignment pattern (dark ring, light ring, dark centre), its template
    drawn through the local affine frame (ex, ey = one module along a row / a column, in pixels), searched ±5
    modules around the predicted centre. dark (H, W) → centre (x, y) and the peak correlation (1 = perfect)."""
    h, w = dark.shape
    module = float(torch.maximum(ex.norm(), ey.norm()))
    half, r = int(2.7 * module) + 1, int(5 * module) + 1
    pad = r + half
    x0, y0 = int(guess[0]) - pad, int(guess[1]) - pad
    if not (-pad <= x0 < w and -pad <= y0 < h):
        return guess, 0.0
    t = torch.arange(-half, half + 1.0)
    dv, du = torch.meshgrid(t, t, indexing="ij")
    XY = torch.linalg.solve(torch.stack([ex, ey], 1), torch.stack([du.reshape(-1), dv.reshape(-1)]))  # code offsets
    ring = XY.abs().max(0).values.reshape(du.shape)  # Chebyshev distance from the centre, in modules
    band = torch.where(ring < 0.5, 1.0, torch.where(ring < 1.5, -1.0, torch.where(ring < 2.5, 1.0, 0.0)))
    band = torch.where(((ring - 0.5).abs() < 0.2) | ((ring - 1.5).abs() < 0.2), 0.0, band)  # blur-tolerant edges
    kernel = band / band.abs().sum()
    padded = F.pad(dark.float() * 2 - 1, (pad, pad, pad, pad), value=-1.0)  # outside the photo counts as light
    patch = padded[y0 + pad : y0 + 3 * pad + 1, x0 + pad : x0 + 3 * pad + 1]
    resp = F.conv2d(patch[None, None], kernel[None, None])[0, 0]  # (2r + 1, 2r + 1)
    yy, xx = torch.meshgrid(torch.arange(2 * r + 1.0), torch.arange(2 * r + 1.0), indexing="ij")
    resp = resp - 0.03 * ((xx - r) ** 2 + (yy - r) ** 2).sqrt() / module  # prefer the nearer of two equal hits
    score = float(resp.max())
    weight = (resp >= score - 0.03).float()
    centre = torch.stack([(xx * weight).sum(), (yy * weight).sum()]) / weight.sum()
    return centre + torch.tensor([x0 + half, y0 + half]), score


def code_points(n: int, version: int) -> torch.Tensor:
    """Code-plane positions (module units) of TL, TR, BL finder centres and of the 4th point (4, 2)."""
    fourth = [n - 3.5, n - 3.5] if version == 1 else [n - 6.5, n - 6.5]
    return torch.tensor([[3.5, 3.5], [n - 3.5, 3.5], [3.5, n - 3.5], fourth])


def homography(dark: torch.Tensor, finders: torch.Tensor, version: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Code plane → image H (3, 3) from the 3 finder centres and a 4th point: the alignment pattern (version ≥ 2),
    found near where the affine map of the finders predicts it, or the parallelogram corner (version 1, as ZXing
    does). Returns H and the four image points."""
    n = tensors.size(version)
    tl, tr, bl = finders
    if version == 1:
        fourth = tr + bl - tl
    else:
        t = (n - 10) / (n - 7)  # affine prediction of the alignment centre at (n − 6.5, n − 6.5)
        guess = tl + t * (tr - tl) + t * (bl - tl)
        fourth, score = find_alignment(dark, guess, (tr - tl) / (n - 7), (bl - tl) / (n - 7))
        if score < 0.3:
            fourth = guess
    img_pts = torch.cat([finders, fourth[None]])
    return eq.dlt(code_points(n, version)[None], img_pts[None])[0], img_pts


def function_template(version: int, quiet: int = 1) -> tuple[torch.Tensor, torch.Tensor]:
    """Known modules of every symbol of this version: finders + separators, timing, alignment, the dark module and a
    ring of `quiet` light quiet-zone modules. → code-plane centres (M, 2) and target values (M,) (+1 dark, −1 light)."""
    val, fun = tensors.function_patterns(version)
    n = tensors.size(version)
    fmt = torch.zeros(n, n, dtype=torch.bool)
    for pos in tensors.format_positions(n):
        fmt[pos[:, 0], pos[:, 1]] = True
    fixed = fun & ~fmt
    fixed[n - 8, 8] = True
    big = torch.zeros(n + 2 * quiet, n + 2 * quiet, dtype=torch.bool)
    big[quiet:-quiet, quiet:-quiet] = fixed
    big_val = torch.zeros_like(big, dtype=torch.long)
    big_val[quiet:-quiet, quiet:-quiet] = val
    ring = torch.ones_like(big)
    ring[quiet:-quiet, quiet:-quiet] = False
    keep = big | ring
    i, j = torch.nonzero(keep, as_tuple=True)
    centres = torch.stack([j, i], -1).float() + 0.5 - quiet  # (X, Y) of module centres
    return centres, (big_val[i, j] * 2 - 1).float()


def refine_homography(dark: torch.Tensor, H: torch.Tensor, version: int, steps: int = 30):
    """Fine-tune H (B, 3, 3) so the known modules land on matching pixels: maximise the correlation
    C = mean_m t_m · d̃(H x_m) between template values t_m (±1) and the smoothed ±1 photo d̃, by Adam on a
    projective correction H = H₀ S⁻¹ (I + Δ) S (S scales the code plane to unit size). Gradients flow through the
    bilinear grid_sample. Coarse to fine: heavy smoothing first (wide basin), then light (sharp optimum). This is
    direct image alignment (Lucas–Kanade style), batched over photos. → refined H (B, 3, 3) and the final
    correlation C (B,) on the sharpest image (≈ 1 when every known module matches, ≈ 0 for a wrong version)."""
    n = tensors.size(version)
    centres, target = function_template(version, quiet=2)
    offs = torch.tensor([[-0.2, -0.2], [0.2, -0.2], [-0.2, 0.2], [0.2, 0.2]])
    pts = (centres[:, None] + offs).reshape(-1, 2).to(H.device)  # 4 samples per module
    target = target.repeat_interleave(4).to(H.device)
    h, w = dark.shape[1:]
    S = torch.diag(torch.tensor([1 / n, 1 / n, 1.0], device=H.device))
    eye = torch.eye(3, device=H.device)
    delta = torch.zeros(H.shape[0], 3, 3, device=H.device, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=3e-3)
    for width in (9, 5, 3):
        soft = eq.box_mean(dark.float() * 2 - 1, width)
        for _ in range(steps):
            Hd = H @ torch.linalg.inv(S) @ (eye + delta) @ S
            uv = eq.apply_homography(Hd, pts.expand(H.shape[0], -1, -1))
            grid = torch.stack([(2 * uv[..., 0] + 1) / w - 1, (2 * uv[..., 1] + 1) / h - 1], -1)[:, None]
            val = F.grid_sample(soft[:, None], grid, mode="bilinear", align_corners=False, padding_mode="border")
            corr = (val[:, 0, 0] * target).mean(1)
            loss = -corr.sum()
            opt.zero_grad()
            loss.backward()
            delta.grad[:, 2, 2] = 0  # the overall scale of H is free: keep it fixed
            opt.step()
    with torch.no_grad():
        Hd = H @ torch.linalg.inv(S) @ (eye + delta) @ S
        return Hd / Hd[:, 2:, 2:], corr.detach()


# --- 4. sample the module grid ---------------------------------------------------------------------------------------


def code_to_grid(H: torch.Tensor, coords: torch.Tensor, size: tuple[int, int]) -> torch.Tensor:
    """grid_sample coordinates in [−1, 1] of the code-plane points (X, Y) on the product grid coords × coords."""
    h, w = size
    Y, X = torch.meshgrid(coords, coords, indexing="ij")
    pts = torch.stack([X, Y], -1).reshape(1, -1, 2).expand(H.shape[0], -1, -1).to(H.device)
    uv = eq.apply_homography(H, pts)  # (B, P, 2) pixel coordinates
    return torch.stack([(2 * uv[..., 0] + 1) / w - 1, (2 * uv[..., 1] + 1) / h - 1], -1)[:, None]


def sample(img: torch.Tensor, H: torch.Tensor, n: int, per_module: int = 3) -> torch.Tensor:
    """Module votes: grid_sample img (B, H, W) at per_module² points in the central 60 % of each module of the code
    plane mapped into the photo by H (B, 3, 3), averaged per module → (B, n, n)."""
    offs = (torch.arange(per_module) + 0.5) / per_module * 0.6 + 0.2  # away from the module edges
    coords = (torch.arange(n)[:, None] + offs[None]).reshape(-1)  # (n·p,)
    grid = code_to_grid(H, coords, img.shape[1:])
    vals = F.grid_sample(img[:, None], grid, mode="bilinear", align_corners=False, padding_mode="border")
    return vals.reshape(-1, n, per_module, n, per_module).mean((2, 4))


def rectify(img: torch.Tensor, H: torch.Tensor, n: int, px: int = 6) -> torch.Tensor:
    """The photo resampled onto the code plane, quiet zone included, px pixels per module. → (B, m·px, m·px)"""
    coords = (torch.arange((n + 2 * tensors.QUIET) * px) + 0.5) / px - tensors.QUIET
    grid = code_to_grid(H, coords, img.shape[1:])
    out = F.grid_sample(img[:, None], grid, mode="bilinear", align_corners=False, padding_mode="border")
    return out.reshape(img.shape[0], len(coords), len(coords))


# --- 5. format, unmask, codewords ------------------------------------------------------------------------------------


def read_format(modules: torch.Tensor) -> tuple[list[str], torch.Tensor, torch.Tensor]:
    """Read both format copies, take the nearest BCH codeword. modules (B, n, n) → ECC level, mask, bit errors."""
    first, second = tensors.format_positions(modules.shape[-1])
    copies = torch.stack([modules[:, first[:, 0], first[:, 1]], modules[:, second[:, 0], second[:, 1]]], 1)
    index, errors = eq.format_decode(copies.reshape(-1, 15))  # (2B,)
    index, errors = index.view(-1, 2), errors.view(-1, 2)
    pick = errors.argmin(1, keepdim=True)
    index, errors = index.gather(1, pick)[:, 0], errors.gather(1, pick)[:, 0]
    from_bits = {b: e for e, b in tensors.ECL_BITS.items()}
    return [from_bits[i >> 3] for i in index.tolist()], index & 7, errors


def read_codewords(modules: torch.Tensor, version: int, masks: torch.Tensor) -> torch.Tensor:
    """XOR the mask back off (it is an involution), read the data modules in placement order, pack 8 bits per byte.
    modules (B, n, n), masks (B,) → codeword stream (B, total)."""
    n = tensors.size(version)
    _, fun = tensors.function_patterns(version)
    pos = tensors.data_positions(fun).to(modules.device)
    pats = torch.stack([eq.mask_pattern(k, n, modules.device) for k in range(8)])[masks].long()  # (B, n, n)
    bits = (modules ^ pats)[:, pos[:, 0], pos[:, 1]][:, : 8 * tensors.TOTAL_CODEWORDS[version]]
    weights = 2 ** torch.arange(7, -1, -1, device=modules.device)
    return (bits.reshape(bits.shape[0], -1, 8) * weights).sum(-1)


def deinterleave(stream: torch.Tensor, version: int, ecl: str) -> torch.Tensor:
    """Undo the column-by-column interleaving. stream (B, total) → blocks (B, nb, k + T)."""
    lay = tensors.layout(version, ecl)
    nb, k, t = lay["blocks"], lay["k"], lay["n_ec"]
    data = stream[:, : nb * k].reshape(-1, k, nb).transpose(1, 2)
    ecc = stream[:, nb * k :].reshape(-1, t, nb).transpose(1, 2)
    return torch.cat([data, ecc], 2)


# --- 6. Reed–Solomon decoding ----------------------------------------------------------------------------------------


def berlekamp_massey(S: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Shortest LFSR Λ(x) = 1 + Λ₁x + … generating the syndromes, for a batch S (B, T) → Λ (B, T + 1), degree L (B,).
    Its roots are the inverse error locators X_k^{-1}."""
    B, T = S.shape
    dev = S.device
    C = torch.zeros(B, T + 1, dtype=torch.long, device=dev)
    C[:, 0] = 1
    prev, L = C.clone(), torch.zeros(B, dtype=torch.long, device=dev)
    m, b = torch.ones(B, dtype=torch.long, device=dev), torch.ones(B, dtype=torch.long, device=dev)
    j = torch.arange(T + 1, device=dev)
    for step in range(T):
        d = eq.gf_sum(eq.gf_mul(C[:, : step + 1], S[:, : step + 1].flip(1)))  # discrepancy ⊕_i Λ_i S_{step−i}
        src = j[None] - m[:, None]
        shifted = torch.where(src >= 0, prev.gather(1, src.clamp(min=0)), 0)  # x^m · B(x)
        updated = C ^ eq.gf_mul(eq.gf_div(d, b)[:, None], shifted)
        change, grow = d != 0, (d != 0) & (2 * L <= step)
        prev = torch.where(grow[:, None], C, prev)
        b = torch.where(grow, d, b)
        L = torch.where(grow, step + 1 - L, L)
        m = torch.where(grow, 1, m + 1)
        C = torch.where(change[:, None], updated, C)
    return C, L


def rs_decode(c: torch.Tensor, n_ec: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Correct up to T/2 byte errors in each block of c (B, N): syndromes → Berlekamp–Massey → Chien search →
    Forney. Returns corrected blocks (B, N), errors fixed (B,), ok (B,) (False = more errors than it can fix)."""
    B, N = c.shape
    dev = c.device
    S = eq.syndromes(c, n_ec)  # (B, T)
    lam, L = berlekamp_massey(S)
    power = N - 1 - torch.arange(N, device=dev)  # position i ↔ locator X_i = α^{N−1−i}
    X, X_inv = eq.gf_pow_alpha(power), eq.gf_pow_alpha(-power)
    at_roots = eq.gf_poly_eval(lam, X_inv) == 0  # Chien search: (B, N) error positions
    # Forney: e_i = X_i · Ω(X_i⁻¹) / Λ'(X_i⁻¹), Ω = S(x) Λ(x) mod x^T, Λ' keeps the odd terms (characteristic 2).
    k = torch.arange(n_ec, device=dev)
    idx = k[:, None] - k[None]  # (T, T): Ω_k = ⊕_{i ≤ k} S_i Λ_{k−i}
    lam_k = torch.where(idx >= 0, lam[:, idx.clamp(min=0)], 0)  # (B, T, T)
    omega = eq.gf_sum(eq.gf_mul(S[:, None, :], lam_k), dim=-1)  # (B, T)
    deriv = torch.where(torch.arange(n_ec, device=dev) % 2 == 0, lam[:, 1:], 0)  # coefficient of x^k in Λ'
    num, den = eq.gf_poly_eval(omega, X_inv), eq.gf_poly_eval(deriv, X_inv)  # (B, N)
    magnitude = eq.gf_mul(X[None].expand(B, -1), eq.gf_div(num, torch.where(den == 0, 1, den)))
    ok = (at_roots.sum(1) == L) & (2 * L <= n_ec)
    fixed = c ^ torch.where(at_roots & ok[:, None], magnitude, 0)
    ok = ok & (eq.syndromes(fixed, n_ec) == 0).all(1)
    return torch.where(ok[:, None], fixed, c), torch.where(ok, L, 0), ok


def parse_bytes(data: torch.Tensor) -> list[str | None]:
    """Byte-mode payload: 4-bit mode (0100), 8-bit length, then bytes shifted by half a byte. data (B, n) → texts."""
    hi, lo = data >> 4, data & 15
    out = []
    for b in range(data.shape[0]):
        length = int(lo[b, 0] << 4 | hi[b, 1])
        if int(hi[b, 0]) != 4 or length > data.shape[1] - 2:
            out.append(None)
            continue
        payload = (lo[b, 1 : 1 + length] << 4) | hi[b, 2 : 2 + length]
        out.append(bytes(payload.tolist()).decode("utf-8", errors="replace"))
    return out


# --- the whole reader ------------------------------------------------------------------------------------------------


def decode(img: torch.Tensor, keep: bool = False) -> list[dict]:
    """Decode a batch of photos (B, H, W). Each result has 'text' (None on failure) and diagnostics; keep=True adds
    the intermediate images used by the animation."""
    dark = binarise(img)
    R, scale = finder_response(dark)
    pooled = F.max_pool2d(R[:, None], (1, 15), stride=1, padding=(0, 7))  # 15 × 15 max filter, separably
    pooled = F.max_pool2d(pooled, (15, 1), stride=1, padding=(7, 0))[:, 0]
    results = [{"text": None, "stage": "finders"} for _ in range(img.shape[0])]
    found = []
    for b in range(len(results)):
        finders, module = pick_finders(peaks(R[b].cpu(), pooled[b].cpu(), scale[b].cpu(), dark[b].cpu()))
        if finders is not None:
            found.append((b, finders))
            span = 0.5 * ((finders[1] - finders[0]).norm() + (finders[2] - finders[0]).norm())
            results[b]["size_estimate"] = float(span) / module + 7  # n ≈ finder distance / module size + 7
    if keep:
        for b, r in enumerate(results):
            r["dark"] = dark[b].cpu()
    if not found:
        return results
    # Try each version: fit H from the finders (+ alignment), refine it against the known modules, and keep the
    # version whose template fits best, with a weak prior from the size estimate n ≈ finder distance / module + 7.
    fits = {b: [] for b, _ in found}  # per photo: (score, version, fields) for each version tried
    for v in (1, 2, 3):
        H0, pts = zip(*[homography(dark[b].cpu(), f, v) for b, f in found])
        H, corr = refine_homography(dark[[b for b, _ in found]], torch.stack(H0).to(img.device), v)
        for (b, f), h0, h, p, c in zip(found, H0, H, pts, corr.tolist()):
            score = c - 0.05 * abs(results[b]["size_estimate"] - tensors.size(v)) / 4
            fits[b].append((score, v, {"version": v, "fit": c, "H0": h0, "H": h, "points": p, "finders": f}))
    for rank in (0, 1):  # best-fitting version first; if Reed–Solomon then fails, retry with the runner-up
        todo = [b for b in fits if rank == 0 or results[b]["text"] is None]
        for b in todo:
            results[b].update({"stage": "format", **sorted(fits[b], key=lambda x: -x[0])[rank][2]})
        for v in (1, 2, 3):
            group = [b for b in todo if results[b]["version"] == v]
            if group:
                decode_group(img, dark, group, v, results, keep)
    return results


def decode_group(img, dark, group: list[int], version: int, results: list[dict], keep: bool):
    """Steps 4–6 for photos that share a version: sample modules, format, unmask, Reed–Solomon, text."""
    n = tensors.size(version)
    H = torch.stack([results[b]["H"] for b in group])
    votes = sample(dark[group].float(), H, n)
    modules = (votes > 0.5).long()  # (G, n, n)
    levels, masks, fmt_err = read_format(modules)
    stream = read_codewords(modules, version, masks)
    for level in set(levels):
        sel = [i for i, e in enumerate(levels) if e == level]
        lay = tensors.layout(version, level)
        blocks = deinterleave(stream[sel], version, level)  # (g, nb, N)
        fixed, n_err, ok = rs_decode(blocks.reshape(-1, blocks.shape[-1]), lay["n_ec"])
        fixed, n_err, ok = fixed.view(blocks.shape), n_err.view(len(sel), -1), ok.view(len(sel), -1)
        data = fixed[..., : lay["k"]].reshape(len(sel), -1)
        texts = parse_bytes(data)
        for i, s in enumerate(sel):
            r = results[group[s]]
            r.update(
                {
                    "stage": "decoded" if ok[i].all() and texts[i] is not None else "reed-solomon",
                    "ecl": level,
                    "mask": int(masks[s]),
                    "format_errors": int(fmt_err[s]),
                    "byte_errors": n_err[i].tolist(),
                    "rs_ok": ok[i].tolist(),
                    "modules": modules[s].cpu(),
                    "received": blocks[i].cpu(),
                    "corrected": fixed[i].cpu(),
                    "text": texts[i] if ok[i].all() else None,
                }
            )
            if keep:
                r["rectified"] = rectify(img[group[s]][None], H[s][None], n)[0].cpu()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/qr_code_decode/plot.py
"""Animate the reader on each photo (photo → binarised → finders + homography → rectified → module errors →
Reed–Solomon fixes and text) next to the success rate against occlusion for the four ECC levels."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import equation as eq
from . import tensors

STAGES = ["photo", "adaptive threshold", "finders + homography", "rectified", "module errors", "Reed–Solomon"]
LEVEL_COLOURS = {"L": "#9ca3af", "M": "#2563eb", "Q": "#16a34a", "H": "#dc2626"}
GREY = plt.get_cmap("gray", 16)  # few grey levels keep the GIF small


def animate(img, results, sym, block, place, message, occs, rates, versions, occlusion, path: Path) -> Path:
    n = sym["grid"].shape[0]
    truth = sym["grid"].numpy()
    events = [(b, s) for b in range(len(results)) for s in range(len(STAGES))]

    fig = plt.figure(figsize=(9.6, 5.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.05], height_ratios=[1, 1.15], wspace=0.25, hspace=0.35)
    ax_img, ax_txt, ax_rate = fig.add_subplot(grid[:, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, 1])

    x = 100 * np.array(occs)
    for level, rate in rates.items():
        ax_rate.plot(x, 100 * rate.numpy(), "o-", ms=4, color=LEVEL_COLOURS[level],
                     label=f"{level} (v{versions[level]})")  # fmt: skip
    ax_rate.axvline(100 * occlusion, color="#f59e0b", ls="--", lw=1.2, label="these photos")
    ax_rate.set_xlabel("occluded share of the symbol (%)")
    ax_rate.set_ylabel("photos decoded (%)")
    ax_rate.set_ylim(-5, 105)
    ax_rate.set_title("success rate per ECC level", fontsize=10)
    ax_rate.legend(fontsize=7, loc="lower left", ncol=2)
    ax_txt.axis("off")
    text = ax_txt.text(0, 1, "", family="monospace", fontsize=8.2, va="top", transform=ax_txt.transAxes)

    def update(f: int):
        b, stage = events[f]
        r = results[b]
        ax_img.clear()
        ax_img.set_axis_off()
        reached = 5 if "modules" in r else (2 if "finders" in r else 1)  # last stage this photo got to
        shown = min(stage, reached)
        if shown == 0:
            ax_img.imshow(img[b].numpy(), cmap=GREY, vmin=0, vmax=1)
        elif shown == 1:
            ax_img.imshow(r["dark"].numpy(), cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
        elif shown == 2:
            ax_img.imshow(img[b].numpy(), cmap=GREY, vmin=0, vmax=1)
            if "finders" in r:
                corners = torch.tensor([[0.0, 0.0], [n, 0.0], [n, n], [0.0, n], [0.0, 0.0]])
                v = r["version"]
                corners = corners * tensors.size(v) / n
                outline = eq.apply_homography(r["H"][None].cpu(), corners[None])[0].numpy()
                ax_img.plot(outline[:, 0], outline[:, 1], color="#2563eb", lw=1.8)
                f_ = r["finders"].numpy()
                ax_img.scatter(f_[:, 0], f_[:, 1], s=140, facecolors="none", edgecolors="#f59e0b", linewidths=2)
                for p, label in zip(f_, ["TL", "TR", "BL"]):
                    ax_img.text(p[0] + 9, p[1] - 9, label, color="#b45309", fontsize=9, weight="bold")
                fourth = r["points"][3].numpy()
                ax_img.scatter([fourth[0]], [fourth[1]], s=60, marker="s", facecolors="none", edgecolors="#dc2626")
        elif shown == 3:
            nv = tensors.size(r["version"])
            ax_img.imshow(r["rectified"].numpy(), cmap=GREY, vmin=0, vmax=1, extent=(-4, nv + 4, nv + 4, -4))
            ax_img.plot([0, nv, nv, 0, 0], [0, 0, nv, nv, 0], color="#2563eb", lw=1)
        else:
            mods = r["modules"].numpy()
            rgb = np.where(mods[..., None] == 1, 0.1, 0.95) * np.ones(3)
            if stage == 4 and mods.shape == truth.shape:
                wrong = mods != truth
                rgb[wrong] = [0.86, 0.15, 0.15]
            elif stage == 5 and mods.shape == truth.shape and r["ecl"] == sym["ecl"]:
                fixed = (r["received"] != r["corrected"]).numpy()  # (nb, N) bytes Reed–Solomon changed
                hit = (block >= 0).numpy() & fixed[block.clamp(min=0).numpy(), place.clamp(min=0).numpy()]
                rgb[hit] = np.where(mods[hit][:, None] == 1, [0.7, 0.35, 0.0], [0.98, 0.75, 0.35])
            ax_img.imshow(rgb, interpolation="nearest")
        names = {1: "binarised by a local box mean", 2: "1:1:3:1:1 finders, DLT outline"}
        title = names.get(shown, STAGES[shown])
        if shown == 4 and r["modules"].shape == truth.shape:
            title = f"sampled modules: {int((r['modules'].numpy() != truth).sum())} wrong (red)"
        if shown == 5:
            title = "bytes fixed by Reed–Solomon (orange)"
        ax_img.set_title(f"photo {b + 1}/{len(results)} · {title}", fontsize=10)

        lines = [f"message : '{message}'", f"symbol  : version {sym['version']}-{sym['ecl']}, mask {sym['mask']}", ""]
        lines.append(f"photo {b + 1}, step {stage + 1}/6: {STAGES[stage]}")
        if "finders" in r and stage >= 2:
            lines.append(f"version from the template fit: {r['version']} (fit {r['fit']:.2f})")
        if "byte_errors" in r and stage >= 3:
            lines.append(f"format: level {r['ecl']}, mask {r['mask']}, {r['format_errors']} bit(s) fixed (BCH)")
        if "byte_errors" in r and stage >= 4:
            t = tensors.layout(r["version"], r["ecl"])["n_ec"] // 2
            fixed = ", ".join(f"{e}/{t}" if ok else f">{t}" for e, ok in zip(r["byte_errors"], r["rs_ok"]))
            lines.append(f"RS byte errors fixed per block: {fixed}")
        if stage == 5 or stage > reached:
            if r["text"] is not None:
                verdict = "correct" if r["text"] == message else "WRONG"
                lines += ["", f"decoded : '{r['text']}'", f"          {verdict}"]
            else:
                lines += ["", f"failed at: {r['stage']}"]
        text.set_text("\n".join(lines))

    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.1)
    return save_gif(fig, update, frame_indices(len(events), 120), path, fps=2)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'message': 'Reed-Solomon 1960',
    'version': 3,
    'ecc': 'M',
    'perspective': 0.1,
    'occlusion': 0.08,
    'noise': 0.05
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.qr_code_decode")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.qr_code_decode.main")
problem_dir = Path("problems/graphics/qr_code_decode")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
